# ASL 数据探索 — Kaggle 远程内核
在 VS Code 右上角选择你已连接的 Kaggle Jupyter Server 的 Python 内核，然后从上到下运行。
本 notebook 独立运行，无需上传 asl.py、读取本地 config.json 或使用本地 .venv。
本页只检查数据并绘图，不启动训练。输出文件将保存到 /kaggle/working/asl-project。


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_DIR = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/asl-project')
print('Python:', sys.executable)
if not INPUT_DIR.is_dir():
    raise RuntimeError('请选择 Kaggle 远程内核；当前环境没有 /kaggle/input。')
print('Input folders:', [p.name for p in INPUT_DIR.iterdir()])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)


In [ ]:
# 如需手动指定，可设为 Path('/kaggle/input/competitions/asl-signs')。
# 默认只搜索几层目录，避免递归扫描近十万个 parquet 文件。
DATA_DIR = None
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            '未找到 ASL 比赛数据。请在 Kaggle notebook 的 Add Input 中添加 '
            'Google - Isolated Sign Language Recognition (asl-signs)，然后重新运行。'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'发现多个数据目录，请在上方手动指定 DATA_DIR: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv 缺少必要列: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv 没有样本')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('类别映射编号必须从零开始且连续')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv 含类别映射中不存在的标签')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def preprocess(frames, length):
    """Official row order: face, left hand, pose, right hand.

    Use xy from both hands. Retain a validity mask; normalize each frame
    relative to the visible hand centroid with a shared xy scale.
    """
    import numpy as np
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    hands, valid = hands[useful], valid[useful]
    if not len(hands):
        return np.zeros((length, 126), dtype=np.float32)
    safe = np.where(valid[..., None], hands, 0)
    count = np.maximum(valid.sum(axis=1), 1)[:, None, None]
    center = safe.sum(axis=1, keepdims=True) / count
    centered = np.where(valid[..., None], safe - center, 0)
    scale = np.sqrt((centered ** 2).sum(axis=(1, 2), keepdims=True) / (count * 2))
    normalized = centered / np.maximum(scale, 1e-6)
    features = np.concatenate([normalized.reshape(len(hands), 84), valid], axis=1)
    # Uniform resampling also repeats frames for short clips.
    indices = np.rint(np.linspace(0, len(features) - 1, length)).astype(int)
    return features[indices].astype(np.float32)


In [ ]:
row = table.iloc[0]
sample_path = DATA_DIR / row['path']
frames = load_frames(sample_path)
features = preprocess(frames, length=32)
print('Sample:', sample_path)
print('Sign:', row['sign'], '| class index:', labels[row['sign']])
print('Raw shape:', frames.shape)
print('Missing fraction:', float(np.isnan(frames).mean()))
print('Processed shape:', features.shape)
assert features.shape == (32, 126)
assert np.isfinite(features).all()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, index in [('left wrist', 468), ('right wrist', 522)]:
    ax.plot(frames[:, index, 0], label=name)
ax.set(xlabel='Frame', ylabel='Wrist x coordinate', title=str(row['sign']))
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'first_sample_wrist_motion.png', dpi=150)
plt.show()

# 显示双手可见关键点最多的一帧。
left = frames[:, 468:489, :2]
right = frames[:, 522:543, :2]
visible = np.isfinite(np.concatenate([left, right], axis=1)).all(axis=-1).sum(axis=1)
frame_index = int(visible.argmax())
fig, ax = plt.subplots(figsize=(6, 5))
for name, hand in [('left hand', left), ('right hand', right)]:
    ax.scatter(hand[frame_index, :, 0], hand[frame_index, :, 1], label=name)
ax.invert_yaxis()
ax.set_aspect('equal', adjustable='datalim')
ax.set(title=f"{row['sign']} | frame {frame_index}", xlabel='x', ylabel='y')
ax.legend()
fig.tight_layout()
plt.show()


## 下一步
确认上面能显示样本、类别数量和手部图。然后再建立按 participant_id 隔离的训练/验证划分，训练小型 CNN。
当前探索 notebook 不会消耗训练 GPU 时间。/kaggle/working 下文件的保留取决于 Kaggle 会话；结束前请下载重要产物或保存带输出的版本。
